# 🧠 ORBIT AI — System Architecture & Workflow Flowchart

This notebook represents the entire clinical-grade BCI workflow, showing how hardware, diagnostic scripts, safety gates, and machine learning models interact in real-time.

## 1. High-Level BCI Ecosystem

```mermaid
graph TD
    subgraph Data Sources / Hardware
        A[BioAmp EXG Pill] -->|Analog Serial| B(bridge_bioamp.py)
        A2[PhysioNet Cloud] -->|EDF File| B2(simulate_tgam.py)
    end

    subgraph Core Network Bridge
        B -->|TCP Port 9999| C((Data Stream Buffer))
        B2 -->|TCP Port 9999| C
    end

    subgraph Offline Engine setup
        T[train_moabb.py] -->|Saves| M[models/moabb_csp_lda.pkl]
        T -->|Saves| S[data/moabb_stats.pkl]
    end

    subgraph Personal Calibration
        CAL[calibrate.py] -->|Reads| C
        CAL -->|Saves| PRO[models/personal_profile.json]
    end

    subgraph Real-Time Dashboard predict_realtime.py
        C -->|Raw Epochs| SG{1. Signal Gate}
        SG --Valid--> WP{2. Warmup Gate}
        SG --Invalid--> SG_REJECT[Status: NO SIGNAL]
        
        WP --Ready--> FM{3. Fatigue Monitor}
        WP --Wait--> WP_W[Status: CALIBRATING]
        
        FM --Safe--> AI[4. CSP+LDA Prediction]
        FM --Critical--> F_REJECT[Status: STAY_IDLE]
        
        AI -->|Raw Probability| SM[5. Adaptive Smoothing]
        SM -->|Weighted Decision| OUT[Virtual Arena Command]
        
        %% Dependencies
        M -.-> AI
        PRO -.-> FM
        PRO -.-> SM
    end
```

## 2. File Index & Responsibilities

| File Name | Purpose | Phase Category |
| :--- | :--- | :--- |
| `config.py` | Constants (Window sizes, Sample rates, File paths). Imported by all. | Global Setup |
| `train_moabb.py` | Connects to standard MOABB datasets, trains the Riemannian CSP+LDA model using cross-validation over 20 subjects. | Offline Learning |
| `simulate_tgam.py` | Downloads medical EDF files and streams continuous 160Hz epochs over TCP. | Signal Simulation |
| `bridge_bioamp.py` | The hardware replacement for the simulator. Reads Serial from Arduino, applies 1-45Hz Butterworth filtering, maps to 64-channel array. | Hardware Interface |
| `calibrate.py` | Guides the user through a 45s recording session. Uses Welch's power spectrum to map personal Theta/Alpha baselines. | Personalization |
| `predict_realtime.py` | The central brain. Houses the 5 layers of safety/processing logic. Manages UI and renders virtual wheelchair commands. | Inference & Execution |


## 3. Real-Time Logic Flow (predict_realtime.py)
```mermaid
sequenceDiagram
    participant S as Streamer (bridge/simulator)
    participant R as predict_realtime.py
    participant W as Wheelchair / UI

    S->>R: JSON (64-channel Array)
    R->>R: 1. is_signal_valid() -> Noisy / Flatline Check
    R->>R: 2. compute_fatigue() -> Theta / (Alpha+Beta)
    R->>R: 3. moabb_pipeline.decision_function()
    R->>R: 4. sigmod(decision) -> [0.0 - 1.0 probability]
    R->>R: 5. Deque Append -> Output = 0.2(P1) + 0.3(P2) + 0.5(P3)
    alt Output > Threshold
        R->>W: Move FORWARD (1)
    else Output <= Threshold
        R->>W: Move IDLE (0)
    end
```